In [ ]:
# Import library
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Import Data
# from google.colab import files   (Colab only)
# uploaded = files.upload()   (Colab only; the data is read from the data folder below)

In [ ]:
# Menampilkan Data
df = pd.read_csv("../data/survey_synthetic.csv")   # synthetic data; the real survey is not published
print("=== Data Awal ===")

print(df.head(), "\n")

# Memastikan kolom
freq = df['Frekuensi']
sev = df['Severitas']

In [ ]:
# Memastikan kolom tersedia
assert "Frekuensi" in df.columns and "Severitas" in df.columns, "Kolom 'Frekuensi' dan 'Severitas' wajib ada!"

freq = df["Frekuensi"].astype(float).values
sev  = df["Severitas"].astype(float).values

print("Jumlah responden:", len(df))
print("Frekuensi min–max:", np.min(freq), "-", np.max(freq))
print("Severitas min–max:", np.min(sev), "-", np.max(sev))
print("\n")

In [ ]:
# Fungsi AIC
def aic(k, loglik):
    return 2*k - 2*loglik

In [ ]:
# Analisis Fitting Distribution Frekuensi
print("=== FITTING DISTRIBUTION FREKUENSI (Poisson, Binomial, Negative Binomial) ===")

freq_results = []

# --- Poisson ---
lam = np.mean(freq)
loglik_pois = np.sum(stats.poisson.logpmf(freq, lam))
aic_pois = aic(1, loglik_pois)
E_pois = lam
try:
    ks_stat_pois, ks_pvalue_pois = stats.kstest(freq, lambda k: stats.poisson.cdf(k, lam))
    pvalue_pois = ks_pvalue_pois
    print(f"Poisson: λ={lam:.4f} | logLik={loglik_pois:.4f} | AIC={aic_pois:.4f} | E[N]={E_pois:.4f} | p-value (KS)={pvalue_pois:.4f}")
except Exception as e:
    pvalue_pois = np.nan
    print(f"Poisson: λ={lam:.4f} | logLik={loglik_pois:.4f} | AIC={aic_pois:.4f} | E[N]={E_pois:.4f} | p-value (KS)=N/A -> {e}")

freq_results.append(["Poisson", f"λ={lam:.4f}", loglik_pois, 1, aic_pois, E_pois, pvalue_pois])


# --- Binomial ---
n_binom = int(np.max(freq))
p_binom = np.mean(freq)/n_binom
loglik_binom = np.sum(stats.binom.logpmf(freq, n_binom, p_binom))
aic_binom = aic(1, loglik_binom)
E_binom = n_binom*p_binom
try:
    ks_stat_binom, ks_pvalue_binom = stats.kstest(freq, lambda k: stats.binom.cdf(k, n_binom, p_binom))
    pvalue_binom = ks_pvalue_binom
    print(f"Binomial: n={n_binom}, p={p_binom:.4f} | logLik={loglik_binom:.4f} | AIC={aic_binom:.4f} | E[N]={E_binom:.4f} | p-value (KS)={pvalue_binom:.4f}")
except Exception as e:
    pvalue_binom = np.nan
    print(f"Binomial: n={n_binom}, p={p_binom:.4f} | logLik={loglik_binom:.4f} | AIC={aic_binom:.4f} | E[N]={E_binom:.4f} | p-value (KS)=N/A -> {e}")
freq_results.append(["Binomial", f"n={n_binom}, p={p_binom:.4f}", loglik_binom, 1, aic_binom, E_binom, pvalue_binom])

# --- Negative Binomial ---
try:
    r_hat, p_hat, loc_hat = stats.nbinom.fit(freq, floc=0)
    loglik_nb = np.sum(stats.nbinom.logpmf(freq, r_hat, p_hat))
    aic_nb = aic(2, loglik_nb)
    E_nb = r_hat * (1 - p_hat) / p_hat
    try:
        ks_stat_nb, ks_pvalue_nb = stats.kstest(freq, lambda k: stats.nbinom.cdf(k, r_hat, p_hat))
        pvalue_nb = ks_pvalue_nb
        print(f"NegBinomial: r={r_hat:.4f}, p={p_hat:.4f} | logLik={loglik_nb:.4f} | AIC={aic_nb:.4f} | E[N]={E_nb:.4f} | p-value (KS)={pvalue_nb:.4f}")
    except Exception as e:
        pvalue_nb = np.nan
        print(f"NegBinomial: r={r_hat:.4f}, p={p_hat:.4f} | logLik={loglik_nb:.4f} | AIC={aic_nb:.4f} | E[N]={E_nb:.4f} | p-value (KS)=N/A -> {e}")
    freq_results.append(["NegBinomial", f"r={r_hat:.4f}, p={p_hat:.4f}", loglik_nb, 2, aic_nb, E_nb, pvalue_nb])
except Exception as e:
    freq_results.append(["NegBinomial", "fit failed", np.nan, np.nan, np.nan, np.nan, np.nan])
    print("NegBinomial: fit failed ->", e)

freq_df = pd.DataFrame(freq_results, columns=["Distribusi", "Parameter", "LogLik", "k", "AIC", "E[N]", "P-value (KS)"])
freq_df = freq_df.sort_values("AIC", ascending=True)
print("\n--- Ringkasan Hasil Fitting Frekuensi ---")
print(freq_df.to_string(index=False))

best_freq = freq_df.iloc[0]
print(f"\n📊 Distribusi Frekuensi Terbaik: {best_freq['Distribusi']} (AIC={best_freq['AIC']:.4f}) | E[N]={best_freq['E[N]']:.4f})")

In [ ]:
# Analisis Fitting Distribution Severitas
print("\n\n=== FITTING DISTRIBUTION SEVERITAS (Exponential, Gamma, Lognormal, Normal) ===")

sev_positive = sev[sev > 0]

sev_results = []

# --- Exponential ---
loc_e, scale_e = stats.expon.fit(sev_positive, floc=0)
loglik_exp = np.sum(stats.expon.logpdf(sev_positive, loc=loc_e, scale=scale_e))
aic_exp = aic(1, loglik_exp)
E_exp = scale_e + loc_e # E[X] = loc + scale
try:
    ad_result_exp = stats.anderson(sev_positive, dist='expon')
    ks_stat_exp, ks_pvalue_exp = stats.kstest(sev_positive, 'expon', args=(loc_e, scale_e))
    pvalue_exp = ks_pvalue_exp
    print(f"Exponential: scale={scale_e:.4f} | logLik={loglik_exp:.4f} | AIC={aic_exp:.4f} | E[X]={E_exp:.2f} | p-value (KS)={pvalue_exp:.4f}")
except Exception as e:
    pvalue_exp = np.nan
    print(f"Exponential: scale={scale_e:.4f} | logLik={loglik_exp:.4f} | AIC={aic_exp:.4f} | E[X]={E_exp:.2f} | p-value (KS)=N/A -> {e}")

sev_results.append(["Exponential", f"scale={scale_e:.4f}", loglik_exp, 1, aic_exp, E_exp, pvalue_exp])

# --- Gamma ---
a_g, loc_g, scale_g = stats.gamma.fit(sev_positive, floc=0)
loglik_g = np.sum(stats.gamma.logpdf(sev_positive, a_g, loc=loc_g, scale=scale_g))
aic_g = aic(2, loglik_g)
E_g = a_g * scale_g + loc_g # E[X] = a*scale + loc
try:
    ks_stat_g, ks_pvalue_g = stats.kstest(sev_positive, 'gamma', args=(a_g, loc_g, scale_g))
    pvalue_g = ks_pvalue_g
    print(f"Gamma: a={a_g:.4f}, scale={scale_g:.4f} | logLik={loglik_g:.4f} | AIC={aic_g:.4f} | E[X]={E_g:.2f} | p-value (KS)={pvalue_g:.4f}")
except Exception as e:
    pvalue_g = np.nan
    print(f"Gamma: a={a_g:.4f}, scale={scale_g:.4f} | logLik={loglik_g:.4f} | AIC={aic_g:.4f} | E[X]={E_g:.2f} | p-value (KS)=N/A -> {e}")
sev_results.append(["Gamma", f"a={a_g:.4f}, scale={scale_g:.4f}", loglik_g, 2, aic_g, E_g, pvalue_g])

# --- Lognormal ---
shape_ln, loc_ln, scale_ln = stats.lognorm.fit(sev_positive, floc=0)
loglik_ln = np.sum(stats.lognorm.logpdf(sev_positive, shape_ln, loc=loc_ln, scale=scale_ln))
aic_ln = aic(2, loglik_ln)
# E[X] = exp(mu + sigma^2/2)
# scale = exp(mu), shape = sigma
E_ln = np.exp(np.log(scale_ln) + 0.5 * shape_ln**2)
try:
    ks_stat_ln, ks_pvalue_ln = stats.kstest(sev_positive, 'lognorm', args=(shape_ln, loc_ln, scale_ln))
    pvalue_ln = ks_pvalue_ln
    print(f"Lognormal: shape={shape_ln:.4f}, scale={scale_ln:.4f} | logLik={loglik_ln:.4f} | AIC={aic_ln:.4f} | E[X]={E_ln:.2f} | p-value (KS)={pvalue_ln:.4f}")
except Exception as e:
    pvalue_ln = np.nan
    print(f"Lognormal: shape={shape_ln:.4f}, scale={scale_ln:.4f} | logLik={loglik_ln:.4f} | AIC={aic_ln:.4f} | E[X]={E_ln:.2f} | p-value (KS)=N/A -> {e}")
sev_results.append(["Lognormal", f"shape={shape_ln:.4f}, scale={scale_ln:.4f}", loglik_ln, 2, aic_ln, E_ln, pvalue_ln])

# --- Normal ---
mu_n, sigma_n = stats.norm.fit(sev)
loglik_n = np.sum(stats.norm.logpdf(sev, mu_n, sigma_n))
aic_n = aic(2, loglik_n)
E_n = mu_n
try:
    ad_result_n = stats.anderson(sev, dist='norm')
    ks_stat_n, ks_pvalue_n = stats.kstest(sev, 'norm', args=(mu_n, sigma_n))
    pvalue_n = ks_pvalue_n
    print(f"Normal: μ={mu_n:.4f}, σ={sigma_n:.4f} | logLik={loglik_n:.4f} | AIC={aic_n:.4f} | E[X]={E_n:.2f} | p-value (KS)={pvalue_n:.4f}")
except Exception as e:
    pvalue_n = np.nan
    print(f"Normal: μ={mu_n:.4f}, σ={sigma_n:.4f} | logLik={loglik_n:.4f} | AIC={aic_n:.4f} | E[X]={E_n:.2f} | p-value (KS)=N/A -> {e}")
sev_results.append(["Normal", f"μ={mu_n:.4f}, σ={sigma_n:.4f}", loglik_n, 2, aic_n, E_n, pvalue_n])


sev_df = pd.DataFrame(sev_results, columns=["Distribusi", "Parameter", "LogLik", "k", "AIC", "E[X]", "P-value (KS)"])
sev_df = sev_df.sort_values("AIC", ascending=True)
print("\n--- Ringkasan Hasil Fitting Severitas ---")
print(sev_df.to_string(index=False))

best_sev = sev_df.iloc[0]
print(f"\n📊 Distribusi Severitas Terbaik: {best_sev['Distribusi']} (AIC={best_sev['AIC']:.4f}) | E[X]={best_sev['E[X]']:.2f})")

In [ ]:
# Analisis Goodness of Fit
# Uji Chi-Square untuk Frekuensi (Binomial)
print("\n=== Chi-Square Test - Frekuensi (Binomial) ===")

best_freq_dist = freq_df.iloc[0]
dist_name = best_freq_dist['Distribusi']

if dist_name == 'Binomial':
    params = best_freq_dist['Parameter']
    n_binom = int(params.split(',')[0].split('=')[1])
    p_binom = float(params.split(',')[1].split('=')[1])

    bins = np.arange(int(np.max(freq)) + 2) - 0.5
    observed_counts, _ = np.histogram(freq, bins=bins)

    unique_freq_values = np.unique(freq)

    observed_series = pd.Series(freq).value_counts().sort_index()
    observed_values = observed_series.index.values
    observed = observed_series.values

    # Menghitung expected probability (probabilitas yang diharapkan) untuk setiap nilai frekuensi yang diamati
    expected_probs = stats.binom.pmf(observed_values, n_binom, p_binom)

    # Menghitung expected counts (jumlah yang diharapkan)
    expected_counts = expected_probs * len(freq)

    # Chi-Square test
    valid_indices = expected_counts > 5

    if np.sum(valid_indices) < 2:
         print("Warning: Too few bins with expected counts > 5 for a reliable Chi-Square test. Proceeding with all bins.")
         chi2_stat, chi2_pvalue = stats.chisquare(f_obs=observed, f_exp=expected_counts)
    else:
        chi2_stat, chi2_pvalue = stats.chisquare(f_obs=observed[valid_indices], f_exp=expected_counts[valid_indices])

    print(f"Chi-Square Statistic: {chi2_stat:.4f}, P-value: {chi2_pvalue:.4f}")

    if chi2_pvalue > 0.05:
        print("Distribusi Frekuensi mendekati Binomial (berdasarkan uji Chi-Square)")
    else:
        print("Distribusi Frekuensi tidak mendekati Binomial (berdasarkan uji Chi-Square)")

else:
    print(f"Uji Chi-Square untuk distribusi {dist_name} belum diimplementasikan atau Binomial bukan distribusi terbaik.")

In [ ]:
# Analisis Goodness of Fit
# Uji KS untuk Severitas
best_sev_dist = sev_df.iloc[0]
dist_name_sev = best_sev_dist['Distribusi']

print(f"\n=== KS Test - Severitas ({dist_name_sev}) ===")

if dist_name_sev == 'Exponential':
    params_sev = best_sev_dist['Parameter']
    scale_exp = float(params_sev.split('=')[1])
    ks_stat_sev, ks_pvalue_sev = stats.kstest(sev_positive, 'expon', args=(0, scale_exp)) # floc=0 for exponential
    print(f"KS Statistic: {ks_stat_sev:.4f}, P-value: {ks_pvalue_sev:.4f}")
    if ks_pvalue_sev > 0.05:
        print("Distribusi Severitas mendekati Eksponensial")
    else:
        print("Distribusi Severitas tidak mendekati Eksponensial")
elif dist_name_sev == 'Lognormal':
    params_sev = best_sev_dist['Parameter']
    shape_ln = float(params_sev.split(',')[0].split('=')[1])
    scale_ln = float(params_sev.split(',')[1].split('=')[1])
    ks_stat_sev, ks_pvalue_sev = stats.kstest(sev_positive, 'lognorm', args=(shape_ln, 0, scale_ln)) # floc=0 for lognormal
    print(f"KS Statistic: {ks_stat_sev:.4f}, P-value: {ks_pvalue_sev:.4f}")
    if ks_pvalue_sev > 0.05:
        print("Distribusi Severitas mendekati Lognormal")
    else:
        print("Distribusi Severitas tidak mendekati Lognormal")
else:
     print(f"Uji KS untuk distribusi {dist_name_sev} belum diimplementasikan.")

In [ ]:
# Menghitung Pure Premium / Premi Murni (berdasarkan ekspektasi distribusi terbaik)
E_freq = best_freq["E[N]"]
E_sev = best_sev["E[X]"]

pure_premium = E_freq * E_sev

print("\n\n=== PERHITUNGAN PURE PREMIUM ===")
print(f"E[N] (ekspektasi klaim per tahun)  = {E_freq:.4f}")
print(f"E[X] (ekspektasi biaya per klaim)   = {E_sev:.2f}")
print(f"👉 Pure Premium (E[N] × E[X])       = {pure_premium:,.2f} per polis per tahun")

total_expected = pure_premium * len(df)
print(f"Total ekspektasi klaim untuk {len(df)} responden = {total_expected:,.2f}\n")


In [ ]:
# Menghitung Gross Premium / Premi Kotor
gross_premium_multiplier = 4/3
gross_premium = pure_premium * gross_premium_multiplier
gpm = gross_premium / 12

print("\n\n=== PERHITUNGAN GROSS PREMIUM ===")
print(f"Faktor Pengali Gross Premium = {gross_premium_multiplier:.4f}")
print(f"👉 Gross Premium (Pure Premium × Faktor Pengali) = {gross_premium:,.2f} per polis per tahun")
print(f"👉 Nilai Premi = {gpm:,.2f} per polis per bulan")

In [ ]:
# Ringkasan Akhir
print("=== RINGKASAN AKHIR ===")
print(f"Frekuensi terbaik: {best_freq['Distribusi']} ({best_freq['Parameter']}) | E[N]={E_freq:.4f}")
print(f"Severitas terbaik: {best_sev['Distribusi']} ({best_sev['Parameter']}) | E[X]={E_sev:.2f}")
print(f"Pure Premium (per polis per tahun) = {pure_premium:,.2f}")
print(f"Gross Premium (per polis per tahun) = {gross_premium:,.2f}")
print(f"Nilai Premi (per polis per bulan) = {gpm:,.2f}")

In [ ]:
# ==========================================================
# VISUALISASI HASIL FITTING DISTRIBUSI FREKUENSI & SEVERITAS
# ==========================================================


In [ ]:
# ==========================================================
# VISUALISASI HASIL FITTING DISTRIBUSI FREKUENSI & SEVERITAS
# ==========================================================

# Plot Distribusi Frekuensi (Histogram + PDF Fitted) ---
plt.figure(figsize=(10, 6))
sns.histplot(freq, kde=False, bins=int(max(freq)+1), color='skyblue', stat='density', label='Data Frekuensi') # Use freq and adjust bins

# Rentang data
x = np.arange(0, int(max(freq)) + 1) # Ensure x is within the bounds of the histogram

# Plot distribusi teoritis berdasarkan hasil fitting
for index, row in freq_df.iterrows():
    dist_name = row['Distribusi']
    if dist_name == "Poisson" and not pd.isna(row['AIC']):
        lam = np.mean(freq)
        plt.plot(x, stats.poisson.pmf(x, lam), 'r-', label=f'Poisson (λ={lam:.4f})')
    elif dist_name == "Binomial" and not pd.isna(row['AIC']):
        n_binom = int(np.max(freq))
        p_binom = np.mean(freq)/n_binom
        plt.plot(x, stats.binom.pmf(x, n_binom, p_binom), 'g--', label=f'Binomial (n={n_binom}, p={p_binom:.4f})')
    elif dist_name == "NegBinomial" and not pd.isna(row['AIC']):
        try:
            r_hat, p_hat, loc_hat = stats.nbinom.fit(freq, floc=0)
            plt.plot(x, stats.nbinom.pmf(x, r_hat, p_hat), 'b-.', label=f'Neg. Binomial (r={r_hat:.4f}, p={p_hat:.4f})')
        except Exception as e:
            print(f"Could not plot NegBinomial: {e}")


plt.title('Distribusi Frekuensi dan Hasil Fitting')
plt.xlabel('Frekuensi Klaim per Tahun')
plt.ylabel('Kepadatan Probabilitas')
plt.legend()
plt.show()

In [ ]:
# Plot Distribusi Severitas (Histogram + PDF Fitted) ---
plt.figure(figsize=(10, 6))
sns.histplot(sev_positive, kde=False, bins=10, color='lightgreen', stat='density', label='Data Severitas (Positif)')

# Mendapatkan parameter yang disesuaikan dari sev_df
params_exp = sev_df[sev_df['Distribusi'] == 'Exponential']['Parameter'].iloc[0]
scale_exp = float(params_exp.split('=')[1])

params_gamma = sev_df[sev_df['Distribusi'] == 'Gamma']['Parameter'].iloc[0]
a_g = float(params_gamma.split(',')[0].split('=')[1])
scale_g = float(params_gamma.split(',')[1].split('=')[1])

params_lognorm = sev_df[sev_df['Distribusi'] == 'Lognormal']['Parameter'].iloc[0]
shape_ln = float(params_lognorm.split(',')[0].split('=')[1])
scale_ln = float(params_lognorm.split(',')[1].split('=')[1])

params_norm = sev_df[sev_df['Distribusi'] == 'Normal']['Parameter'].iloc[0]
mu_n = float(params_norm.split(',')[0].split('=')[1])
sigma_n = float(params_norm.split(',')[1].split('=')[1])

x = np.linspace(sev_positive.min(), sev_positive.max(), 1000)

plt.plot(x, stats.expon.pdf(x, loc=0, scale=scale_exp), 'b-.', label=f'Exponential (scale={scale_exp:.4f})')
plt.plot(x, stats.gamma.pdf(x, a_g, loc=0, scale=scale_g), 'r-', label=f'Gamma (a={a_g:.4f}, scale={scale_g:.4f})')
plt.plot(x, stats.lognorm.pdf(x, shape_ln, loc=0, scale=scale_ln), 'g--', label=f'Lognormal (shape={shape_ln:.4f}, scale={scale_ln:.4f})')

x_norm = np.linspace(sev.min(), sev.max(), 1000)
plt.plot(x_norm, stats.norm.pdf(x_norm, mu_n, sigma_n), 'k:', label=f'Normal (μ={mu_n:.4f}, σ={sigma_n:.4f})')

plt.title('Distribusi Severitas dan Hasil Fitting')
plt.xlabel('Biaya Perbaikan per Klaim (Rupiah)')
plt.ylabel('Kepadatan Probabilitas')
plt.legend()
plt.show()

In [ ]:
# Scatter Plot Frekuensi vs Severitas ---
plt.figure(figsize=(8, 5))
sns.scatterplot(x=freq, y=sev)
plt.title('Hubungan antara Frekuensi dan Severitas')
plt.xlabel('Frekuensi Klaim per Tahun')
plt.ylabel('Biaya Perbaikan (Severitas)')
plt.grid(True)
plt.show()

In [ ]:
# Boxplot untuk Sebaran Data
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(y=freq, ax=ax[0], color='skyblue')
ax[0].set_title('Sebaran Data Frekuensi')
sns.boxplot(y=sev, ax=ax[1], color='lightgreen')
ax[1].set_title('Sebaran Data Severitas')
plt.show()